In [ ]:
# ETL - Extract Transform Load
# Extract - get data from outside
# transfrom -cleaning of data
# loab - load the data sql Database 


In [2]:
# extract - get data from files
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Tejas',
    database='shopdb'
)

# create dataframes for CSV files
products_df = pd.read_csv(r"C:\\Users\\hp\\Downloads\\Telegram Desktop\\products_data.csv")

users_df = pd.read_csv(r"C:\\Users\\hp\\Downloads\\Telegram Desktop\\users_data.csv")

orders_df = pd.read_csv(r"C:\\Users\\hp\\Downloads\\Telegram Desktop\\orders_data.csv")

# create dataframes for Excel files
payments_df = pd.read_excel(
    r"C:\\Users\\hp\Downloads\\Telegram Desktop\\payments_data.xlsx",
    engine='openpyxl'
)

reviews_df = pd.read_excel(
    r"C:\\Users\\hp\\Downloads\\Telegram Desktop\\reviews_data.xlsx",
    engine='openpyxl'
)

print("Files loaded successfully")

Files loaded successfully


In [ ]:
# transform - cleaning of data

import pandas as pd

# remove extra spaces from column names
products_df.columns = products_df.columns.str.strip()
users_df.columns = users_df.columns.str.strip()
orders_df.columns = orders_df.columns.str.strip()
payments_df.columns = payments_df.columns.str.strip()
reviews_df.columns = reviews_df.columns.str.strip()

# fill null values
products_df = products_df.fillna({
    'description': 'No Description',
    'Category': 'Miscellaneous'
})

orders_df = orders_df.fillna({
    'TotalAmount': 0.0
})

payments_df = payments_df.fillna({
    'PaymentDate': pd.Timestamp('1970-01-01'),
    'Amount': 0.0
})

reviews_df = reviews_df.fillna({
    'ReviewText': 'No Review Available'
})

# function to generate email id
def generate_unique_email(index):
    return f'unknown{index}@example.com'

# fix invalid emails
users_df['Email'] = users_df.apply(
    lambda row:
    row['Email']
    if '@' in str(row['Email']) and '.' in str(row['Email'])
    else generate_unique_email(row.name),
    axis=1
)

# convert Price column to numeric
products_df['Price'] = pd.to_numeric(
    products_df['Price'],
    errors='coerce'
).fillna(0.0).round(2)

# convert StockQuantity column
products_df['StockQuantity'] = pd.to_numeric(
    products_df['StockQuantity'],
    errors='coerce'
).fillna(0)

# convert OrderDate column
orders_df['OrderDate'] = pd.to_datetime(
    orders_df['OrderDate'],
    errors='coerce',
    format='%Y-%m-%d'
).fillna(pd.Timestamp('1970-01-01'))

# format date
orders_df['OrderDate'] = orders_df['OrderDate'].dt.strftime('%Y-%m-%d')

# remove duplicate rows
products_df = products_df.drop_duplicates()
users_df = users_df.drop_duplicates()
orders_df = orders_df.drop_duplicates()
payments_df = payments_df.drop_duplicates()
reviews_df = reviews_df.drop_duplicates()

print("Data cleaning completed successfully")

In [ ]:
# Delete old records
tables_to_clear = ['payments', 'reviews', 'orders', 'users', 'products']

for table in tables_to_clear:
    cursor = conn.cursor()

    for table in tables_to_clear:
        cursor.execute(f"DELETE FROM `{table}`")

    conn.commit()

conn.commit()

# =========================
# FIX NaN VALUES
# =========================

import numpy as np

products_df = products_df.replace({np.nan: None})
users_df = users_df.replace({np.nan: None})
orders_df = orders_df.replace({np.nan: None})
payments_df = payments_df.replace({np.nan: None})
reviews_df = reviews_df.replace({np.nan: None})

# =========================
# INSERT PRODUCTS
# =========================

product_id_mapping = {}

for index, row in products_df.iterrows():

    cursor.execute("""
        INSERT INTO products
        (product_name, p_description, price, stock_quantity, category)
        VALUES (%s, %s, %s, %s, %s)
    """,
    (
        row['ProductName'],
        row['Description'],
        row['Price'],
        row['StockQuantity'],
        row['Category']
    ))

    product_id_mapping[index] = cursor.lastrowid

conn.commit()

# =========================
# INSERT USERS
# =========================

user_id_mapping = {}

for index, row in users_df.iterrows():

    cursor.execute("""
        INSERT INTO users
        (username, email, address, u_password)
        VALUES (%s, %s, %s, %s)
    """,
    (
        row['UserName'],
        row['Email'],
        row['Address'],
        row['Password']
    ))

    user_id_mapping[index + 1] = cursor.lastrowid

conn.commit()

# =========================
# INSERT ORDERS
# =========================

orders_df['UserID'] = orders_df['UserID'].map(user_id_mapping)
orders_df['UserID'] = orders_df['UserID'].fillna(0).astype(int)

# remove invalid ids
orders_df = orders_df[orders_df['UserID'] != 0]

order_id_mapping = {}

for index, row in orders_df.iterrows():

    cursor.execute("""
        INSERT INTO orders
        (userid, order_date, total_amt, email)
        VALUES (%s, %s, %s, %s)
    """,
    (
        row['UserID'],
        row['OrderDate'],
        row['TotalAmount'],
        row['Email']
    ))

    order_id_mapping[index] = cursor.lastrowid

conn.commit()

# =========================
# INSERT PAYMENTS
# =========================

payments_df['OrderID'] = payments_df['OrderID'].map(order_id_mapping)
payments_df['OrderID'] = payments_df['OrderID'].fillna(0).astype(int)

# remove invalid ids
payments_df = payments_df[payments_df['OrderID'] != 0]

for index, row in payments_df.iterrows():

    cursor.execute("""
        INSERT INTO payments
        (orderid, payment_method, payment_date, amount)
        VALUES (%s, %s, %s, %s)
    """,
    (
        row['OrderID'],
        row['PaymentMethod'],
        row['PaymentDate'],
        row['Amount']
    ))

conn.commit()

# =========================
# INSERT REVIEWS
# =========================

reviews_df['UserID'] = reviews_df['UserID'].map(user_id_mapping)
reviews_df['ProductID'] = reviews_df['ProductID'].map(product_id_mapping)

reviews_df['UserID'] = reviews_df['UserID'].fillna(0).astype(int)
reviews_df['ProductID'] = reviews_df['ProductID'].fillna(0).astype(int)

# convert rating
reviews_df['Rating'] = pd.to_numeric(
    reviews_df['Rating'],
    errors='coerce'
)

# remove invalid rows
reviews_df = reviews_df[
    (reviews_df['UserID'] != 0) &
    (reviews_df['ProductID'] != 0)
]

reviews_df = reviews_df.dropna(subset=['Rating'])

for index, row in reviews_df.iterrows():

    cursor.execute("""
        INSERT INTO reviews
        (productid, userid, rating, review_text)
        VALUES (%s, %s, %s, %s)
    """,
    (
        row['ProductID'],
        row['UserID'],
        row['Rating'],
        row['ReviewText']
    ))

conn.commit()

print("Data inserted successfully")


NameError: name 'cursor' is not defined

In [ ]:
# verify that correct number of records are added to database

def verify_record_count(table_name, expected_count):

    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")

    count = cursor.fetchone()[0]

    # fetchone() -> gets one record
    # [0] -> first column value

    if count == expected_count:
        print(f"{table_name} verification passed: {count} records")

    else:
        print(f"{table_name} verification failed: {count} records")


# verify tables

verify_record_count('products', len(products_df))

verify_record_count('users', len(users_df))

verify_record_count('orders', len(orders_df))

verify_record_count('payments', len(payments_df))

verify_record_count('reviews', len(reviews_df))